In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("HDFS_Partitioning_Benchmark") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000") \
    .getOrCreate()

print(spark)

In [4]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("hdfs://namenode:9000/2023_Yellow_Taxi_Trip_Data.csv")

print("Lecture reussie")

Lecture reussie


In [6]:
print(f"Nombre de lignes: {df.count()}")

Nombre de lignes: 4843349


In [8]:
df.show(5, truncate=False)


+--------+----------------------+----------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime  |tpep_dropoff_datetime |passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+----------------------+----------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|2       |01/01/2023 12:32:10 AM|01/01/2023 12:40:36 AM|1              |0.97         |1         |N                 |161         |141         |2           |9.3        |1    |0.5  

In [10]:
from pyspark.sql.functions import col, to_timestamp, month

df_casted = df.withColumn("Pickup_Datetime", 
    to_timestamp(col("tpep_pickup_datetime"), "MM/dd/yyyy hh:mm:ss a"))

df_final = df_casted.withColumn("Month", month(col("Pickup_Datetime"))) \
                    .filter(col("Month").isNotNull() & col("VendorID").isNotNull())

print("Colonnes Pickup_Datetime et Month creees")
df_final.select("VendorID", "tpep_pickup_datetime", "Month").show(5)

Colonnes Pickup_Datetime et Month creees
+--------+--------------------+-----+
|VendorID|tpep_pickup_datetime|Month|
+--------+--------------------+-----+
|       2|01/01/2023 12:32:...|    1|
|       2|01/01/2023 12:55:...|    1|
|       2|01/01/2023 12:25:...|    1|
|       1|01/01/2023 12:03:...|    1|
|       2|01/01/2023 12:10:...|    1|
+--------+--------------------+-----+
only showing top 5 rows



In [12]:
df_final.write.mode("overwrite").parquet("hdfs://namenode:9000/data/taxi_flat.parquet")
print("Parquet plat sauvegarde sur HDFS")

Parquet plat sauvegarde sur HDFS


In [16]:
!docker exec -it namenode hdfs dfs -ls /data/taxi_flat.parquet

/bin/sh: 1: docker: not found


In [18]:
df_final.write.mode("overwrite") \
    .partitionBy("VendorID", "Month") \
    .parquet("hdfs://namenode:9000/data/taxi_partitioned.parquet")
print("Parquet partitionne cree")

Parquet partitionne cree


In [21]:
from pyspark.sql.functions import col, month, to_timestamp
import time

print(" Benchmark 1: CSV Brut ")

df_csv = spark.read.option("header", "true") \
    .csv("hdfs://namenode:9000/2023_Yellow_Taxi_Trip_Data.csv")

df_csv = df_csv.withColumn("Pickup_Timestamp", 
    to_timestamp(col("tpep_pickup_datetime"), "MM/dd/yyyy hh:mm:ss a"))

df_csv = df_csv.withColumn("Month", month(col("Pickup_Timestamp")))

start = time.time()
count_csv = df_csv.filter((col("VendorID") == "1") & (col("Month") == 1)).count()
end = time.time()

print(f"Lignes trouvees: {count_csv}")
print(f"Temps: {end - start:.4f} secondes")

 Benchmark 1: CSV Brut 
Lignes trouvees: 827367
Temps: 2.9592 secondes


In [23]:
print("Benchmark 2: Parquet Plat ")

df_flat = spark.read.parquet("hdfs://namenode:9000/data/taxi_flat.parquet")

start = time.time()
count_flat = df_flat.filter((col("VendorID") == 1) & (col("Month") == 1)).count()
end = time.time()

print(f"Lignes trouvees: {count_flat}")
print(f"Temps: {end - start:.4f} secondes")

Benchmark 2: Parquet Plat 
Lignes trouvees: 827367
Temps: 0.4295 secondes


In [25]:
print(" Benchmark 3: Parquet Partitionne ")

df_part = spark.read.parquet("hdfs://namenode:9000/data/taxi_partitioned.parquet")

start = time.time()
count_part = df_part.filter((col("VendorID") == 1) & (col("Month") == 1)).count()
end = time.time()

print(f"Lignes trouvees: {count_part}")
print(f"Temps: {end - start:.4f} secondes")

 Benchmark 3: Parquet Partitionne 
Lignes trouvees: 827367
Temps: 0.2863 secondes
